In [1]:
import os 
from dotenv import load_dotenv,find_dotenv
_= load_dotenv(find_dotenv())
groq_api_key = os.environ['GROQ_API_KEY']

In [2]:
from langchain_groq import ChatGroq
llm = ChatGroq(model='openai/gpt-oss-20b')

/opt/miniconda3/envs/llmapp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_community.tools.tavily_search import TavilySearchResults
search_result = TavilySearchResults(max_results = 3)

search_result.invoke("who is the james bond")

/var/folders/z8/cxl79vz558q3_ts3_l1n5zj00000gn/T/ipykernel_9273/3401030312.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
/var/folders/z8/cxl79vz558q3_ts3_l1n5zj00000gn/T/ipykernel_9273/3401030312.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_result = TavilySearchResults(max_results = 3)


[{'title': 'James Bond (literary character) - Wikipedia',
  'url': 'https://en.wikipedia.org/wiki/James_Bond_(literary_character)',
  'content': 'Commander "Commander (Royal Navy)") James Bond CMG RNVR is a character created by the British journalist and novelist Ian Fleming in 1953. He is the protagonist of the James Bond series of novels, films, comics "James Bond (comics)") and video games. Fleming wrote twelve Bond novels and two short story collections. His final two books—The Man with the Golden Gun "The Man with the Golden Gun (novel)") (1965) and Octopussy and The Living Daylights (1966)—were published posthumously. [...] The character is a Secret Service officer, code number 007 (pronounced "double-O[/oʊ/]-seven"), residing in London but active internationally. Bond was a composite character who was based on a number of commandos whom Fleming knew during his service in the Naval Intelligence Division "Naval Intelligence Division (United Kingdom)") during the Second World War, 

In [4]:
tools = [search_result]

In [5]:
from langchain.agents import create_agent

agent_executor = create_agent(llm,tools)

In [6]:
from langchain_core.messages import HumanMessage

response = agent_executor.invoke({"messages":[HumanMessage(content='what is the latest news?')]})

response["messages"]

[HumanMessage(content='what is the latest news?', additional_kwargs={}, response_metadata={}, id='b9dda0b5-ba58-45ce-9feb-f59be423165b'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks "what is the latest news?" We need current news. We should use the search tool.', 'tool_calls': [{'id': 'fc_be867eef-b75c-4048-9b36-4a4e1d090c7d', 'function': {'arguments': '{"query":"latest news"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 160, 'total_tokens': 211, 'completion_time': 0.055533109, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.007789471, 'prompt_tokens_details': None, 'queue_time': 0.156630157, 'total_time': 0.06332258}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0488c-275a-7322-b447-f48

# Adding memory

In [7]:
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent
memory = MemorySaver()

agent_executor = create_agent(llm,tools,checkpointer = memory)
config  = {"configurable": {"thread_id":"001"}} #used to isolate the conversations between multiple users using thread id 

In [8]:
for chunk in agent_executor.stream({"messages":[HumanMessage(content="who won the 2024 soccer eurocup?")]},config):
    print(chunk)
    print('===========================')

{'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "who won the 2024 soccer eurocup?" We need to check if the Euro 2024 tournament has concluded. Euro 2024 is scheduled to be held in Germany, from June 14 to July 14, 2024. As of current date (August 28, 2026), the tournament would have occurred. So we need to provide the winner. I need to confirm the winner: I recall that Spain won Euro 2024? Actually, I recall that Spain won Euro 2024? Wait, I think Spain won Euro 2024? I need to check. Let\'s search.', 'tool_calls': [{'id': 'fc_14e2563c-b706-4ac5-8646-8d5d45f49891', 'function': {'arguments': '{"query":"Euro 2024 champion"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 161, 'prompt_tokens': 164, 'total_tokens': 325, 'completion_time': 0.201721336, 'completion_tokens_details': {'reasoning_tokens': 130}, 'prompt_time': 0.011998615, 'prompt_tokens_details': None, '

In [9]:
for chunk in agent_executor.stream({"messages":[HumanMessage(content="who were the top stars of the winner team")]},config):
    print(chunk)
    print('===========================')

{'model': {'messages': [AIMessage(content='', additional_kwargs={'reasoning_content': "Need to list top stars of Spain's squad. Provide key players: Lionel? Actually Spanish stars: Dani Carvajal, Sergio Busquets, Pedri, Gavi, Ferran Torres, Ferran? Actually Ferran Torres is Spain but also in England. Other stars: Ferran? Actually Ferran Torres is a Spanish striker, yes. Also: Ferran? Wait: Ferran Torres is Spanish. Also: Ferran? Might mention: Ferran Torres, Pedri, Gavi, Dani Carvajal, Sergio Busquets, Ferran? Actually Ferran Torres is Spanish. Also: Ferran? Let's recall: Spain's squad: Goalkeeper: Unai Simón. Defenders: Dani Carvajal, Sergio Ramos (though retired? He was not in squad), Gerard Piqué? Actually Piqué retired. But in Euro 2024, Spain had new players: Dani Carvajal, Sergio Ramos? He retired earlier. They had Dani Carvajal, Sergio Ramos? Wait: In Euro 2024, Spain had Dani Carvajal, Sergio Ramos? He was not there. They had Dani Carvajal, Sergio Ramos? Let's check. Actually S

* based on the thread id the model dosent know the below questions context

In [10]:
config  = {"configurable": {"thread_id":"002"}}
for chunk in agent_executor.stream({"messages":[HumanMessage(content="who were the top stars of the winner team")]},config):
    print(chunk)
    print('===========================')

{'model': {'messages': [AIMessage(content='I’m not sure which event or team you’re referring to. Could you let me know which tournament or sport you have in mind? That way I can give you the correct list of top stars from the winning team.', additional_kwargs={'reasoning_content': 'We need context: "winner team" refers to some event. Likely a sports tournament. The user asks: "who were the top stars of the winner team". Without context, we need to infer from prior conversation? There\'s no prior context. We need to ask clarifying question. But maybe it\'s referencing a known tournament: e.g., FIFA World Cup, NBA, etc. Could be about the 2023 World Cup? The winner team: Argentina? Top stars: Lionel Messi, etc. But the user didn\'t specify. Could be about the 2023 Cricket World Cup? Winner: India? Top stars: Rohit Sharma, etc. Or maybe about a recent sports event. Without context, we can ask for clarification. According to policy, we should ask for clarification.'}, response_metadata={'t